# Part 3 — Recursive Function Simulation
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---

### How to use this notebook
Run each cell top-to-bottom with **Shift+Enter**.

| Cell | What it does |
|------|-------------|
| **Cell 1** | Factorial, Fibonacci (naive + memoized), Tower of Hanoi |
| **Cell 2** | GCD, Fast Power, Binary Search |
| **Cell 3** | Call stack profiler — measures actual calls and depth |
| **Cell 4** | Growth charts — visualises O(n) vs O(2ⁿ) explosions |
| **Cell 5** | Automated test suite |

---

### What is Recursion?
A function that calls itself. Every recursive function needs:
1. **Base case** — stops the recursion, returns without calling itself
2. **Recursive case** — reduces the problem closer to the base case

**Data structure: The Call Stack**  
Each function call pushes a **stack frame** (local variables, return address).  
When the function returns, the frame is popped. This is LIFO (Last In, First Out).

```
factorial(4)  ← push frame
  factorial(3)  ← push frame
    factorial(2)  ← push frame
      factorial(1)  ← push frame  (base case)
      return 1      ← pop frame
    return 2×1=2    ← pop frame
  return 3×2=6      ← pop frame
return 4×6=24       ← pop frame
```

---

### Flowcharts

**Factorial**
```
factorial(n):
    if n <= 1:  return 1                 ← base case
    return n × factorial(n - 1)          ← recursive case
```

**Fibonacci (Naive)**
```
fibonacci(n):
    if n <= 1:  return n                 ← base case
    return fibonacci(n-1) + fibonacci(n-2)  ← two recursive calls!
    ⚠️  Exponential O(2^n) calls — recomputes same subproblems many times
```

**Fibonacci (Memoized)**
```
fibonacci(n, memo={}):
    if n in memo:  return memo[n]        ← return cached result (O(1)!)
    if n <= 1:     return n
    memo[n] = fibonacci(n-1) + fibonacci(n-2)
    return memo[n]                       ← only O(n) unique calls
```

**Tower of Hanoi**
```
hanoi(n, src, aux, dst):
    if n == 1:
        print(f"Move disk 1 from {src} to {dst}")   ← base case
        return
    hanoi(n-1, src, dst, aux)           ← move top n-1 to aux
    print(f"Move disk {n} from {src} to {dst}")
    hanoi(n-1, aux, src, dst)           ← move n-1 from aux to dst

Total moves = 2^n - 1
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 1 — FACTORIAL, FIBONACCI, TOWER OF HANOI
#  Output format matches the project spec exactly.
#  Each function traces its own call tree to stdout.
# ═══════════════════════════════════════════════════════════════════════
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 110,
    "font.family": "monospace",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})

def _pad(depth):
    return "  " * depth        # indentation for call tree display


# ════════════════════════════════════════════════════════════════════════
#  FACTORIAL
#  Complexity: O(n) calls, O(n) max stack depth
#  Data structure: call stack grows to depth n, then fully unwinds
# ════════════════════════════════════════════════════════════════════════

def factorial(n, depth=0, _log=None):
    if _log is None:
        _log = []
        if not (isinstance(n, int) and n >= 0):
            raise ValueError(f"factorial needs a non-negative int, got {n!r}")

    _log.append(f"{_pad(depth)}factorial({n})")

    if n <= 1:
        _log.append(f"{_pad(depth + 1)}-> base case: return 1")
        return 1, _log

    sub, _ = factorial(n - 1, depth + 1, _log)
    result  = n * sub
    _log.append(f"{_pad(depth)}-> {n} * factorial({n-1}) = {result}")
    return result, _log


def run_factorial(n):
    print("─" * 50)
    print(f"  factorial({n})")
    print("─" * 50)
    result, log = factorial(n)
    for line in log:
        print(f"  {line}")
    print(f"  Result = {result}")
    print("─" * 50)


# ════════════════════════════════════════════════════════════════════════
#  FIBONACCI
#  Naive    : O(2^n) calls — recomputes same values repeatedly
#  Memoized : O(n) calls  — stores results in a dict cache
# ════════════════════════════════════════════════════════════════════════

def fibonacci(n, depth=0, _log=None, _memo=None, use_memo=False):
    if _log  is None: _log  = []
    if _memo is None: _memo = {}
    if not (isinstance(n, int) and n >= 0):
        raise ValueError(f"fibonacci needs a non-negative int, got {n!r}")

    # memoization check — return cached result immediately
    if use_memo and n in _memo:
        _log.append(f"{_pad(depth)}fibonacci({n})  ← [cache hit] = {_memo[n]}")
        return _memo[n], _log

    _log.append(f"{_pad(depth)}fibonacci({n})")

    if n <= 1:
        _log.append(f"{_pad(depth + 1)}-> base case: return {n}")
        if use_memo: _memo[n] = n
        return n, _log

    l, _ = fibonacci(n - 1, depth + 1, _log, _memo, use_memo)
    r, _ = fibonacci(n - 2, depth + 1, _log, _memo, use_memo)
    result = l + r
    if use_memo: _memo[n] = result
    _log.append(f"{_pad(depth)}-> fib({n-1}) + fib({n-2}) = {l} + {r} = {result}")
    return result, _log


def run_fibonacci(n, use_memo=False):
    mode = "(Memoized ⚡)" if use_memo else "(Naive)"
    print("─" * 52)
    print(f"  fibonacci({n})  {mode}")
    print("─" * 52)
    result, log = fibonacci(n, use_memo=use_memo)
    for line in log:
        print(f"  {line}")
    print(f"  Result = {result}")
    print("─" * 52)


# ════════════════════════════════════════════════════════════════════════
#  TOWER OF HANOI
#  Total moves = 2^n - 1  (proven by induction)
#  Data structure: implicit call stack + explicit moves list
# ════════════════════════════════════════════════════════════════════════

def tower_of_hanoi(n, src="A", aux="B", dst="C", depth=0, _log=None, _moves=None):
    if _log   is None: _log   = []
    if _moves is None:
        _moves = []
        if not (isinstance(n, int) and n >= 1):
            raise ValueError(f"n must be a positive integer, got {n!r}")

    _log.append(f"{_pad(depth)}hanoi({n}, {src}→{dst} via {aux})")

    if n == 1:
        move = f"Move disk 1 from {src} to {dst}"
        _moves.append(move)
        _log.append(f"{_pad(depth + 1)}-> {move}")
        return _moves, _log

    tower_of_hanoi(n - 1, src, dst, aux, depth + 1, _log, _moves)
    move = f"Move disk {n} from {src} to {dst}"
    _moves.append(move)
    _log.append(f"{_pad(depth)}-> {move}")
    tower_of_hanoi(n - 1, aux, src, dst, depth + 1, _log, _moves)
    return _moves, _log


def run_hanoi(n):
    print("─" * 55)
    disk_label = "disks" if n > 1 else "disk"
    print(f"  Tower of Hanoi  ({n} {disk_label})")
    print("─" * 55)
    moves, log = tower_of_hanoi(n)

    if n <= 3:                     # print full call tree for small n
        print("  Call tree:")
        for line in log:
            print(f"    {line}")
        print()

    print("  Move sequence:")
    for i, m in enumerate(moves, 1):
        print(f"    {i:>3}.  {m}")
    print()
    print(f"  Total moves     : {len(moves)}")
    formula = 2**n - 1
    verified = '✓' if len(moves) == formula else '✗'
    print(f"  Formula (2^n-1) : {formula}")
    print(f"  Verified        : {verified}")
    print("─" * 55)


# ── Run all three ────────────────────────────────────────────────────
print("=" * 55)
print("  REQUIRED RECURSIVE FUNCTIONS")
print("=" * 55)
print()

run_factorial(6)
print()
run_fibonacci(6, use_memo=False)   # naive
print()
run_fibonacci(6, use_memo=True)    # memoized — compare call count!
print()
run_hanoi(4)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 2 — EXTENDED RECURSIVE FUNCTIONS (bonus)
#  GCD (Euclidean), Fast Power, Binary Search
# ═══════════════════════════════════════════════════════════════════════

# ── GCD — Euclidean Algorithm ──────────────────────────────────────────
# gcd(a, b) = gcd(b, a mod b)  until b = 0
# Complexity: O(log min(a, b)) depth
def gcd(a, b, depth=0, _log=None):
    if _log is None:
        _log = []
        if not (isinstance(a,int) and isinstance(b,int) and a>=0 and b>=0):
            raise ValueError("gcd requires non-negative integers")

    _log.append(f"{_pad(depth)}gcd({a}, {b})")

    if b == 0:
        _log.append(f"{_pad(depth + 1)}-> base case: return {a}")
        return a, _log

    r, _ = gcd(b, a % b, depth + 1, _log)
    _log.append(f"{_pad(depth)}-> gcd({b}, {a}%{b}={a%b}) = {r}")
    return r, _log


# ── Fast Power (Exponentiation by Squaring) ───────────────────────────
# base^exp = (base^(exp//2))^2   if exp is even
#          = base × base^(exp-1) if exp is odd
# Complexity: O(log n) depth — far fewer multiplications than naive
def fast_power(base, exp, depth=0, _log=None):
    if _log is None:
        _log = []
        if not (isinstance(exp, int) and exp >= 0):
            raise ValueError("exp must be a non-negative integer")

    _log.append(f"{_pad(depth)}power({base}, {exp})")

    if exp == 0:
        _log.append(f"{_pad(depth + 1)}-> base case: return 1")
        return 1, _log

    if exp % 2 == 0:
        half, _ = fast_power(base, exp // 2, depth + 1, _log)
        result   = half * half
        _log.append(f"{_pad(depth)}-> (power({base},{exp//2}))² = {half}² = {result}")
    else:
        sub, _  = fast_power(base, exp - 1, depth + 1, _log)
        result   = base * sub
        _log.append(f"{_pad(depth)}-> {base} × power({base},{exp-1}) = {result}")

    return result, _log


# ── Binary Search ─────────────────────────────────────────────────────
# Halves the search space each call. O(log n) depth.
def binary_search(arr, target, lo=0, hi=None, depth=0, _log=None):
    if _log is None: _log = []
    if hi   is None: hi   = len(arr) - 1

    _log.append(f"{_pad(depth)}binary_search([{lo}..{hi}], target={target})")

    if lo > hi:
        _log.append(f"{_pad(depth + 1)}-> not found")
        return -1, _log

    mid = (lo + hi) // 2
    _log.append(f"{_pad(depth + 1)}-> mid={mid}, arr[{mid}]={arr[mid]}")

    if arr[mid] == target:
        _log.append(f"{_pad(depth + 1)}-> FOUND at index {mid} ✓")
        return mid, _log

    if arr[mid] < target:
        return binary_search(arr, target, mid+1, hi, depth+1, _log)
    return binary_search(arr, target, lo, mid-1, depth+1, _log)


# ── Run demos ──────────────────────────────────────────────────────────
print("=" * 52)
print("  GCD(48, 18)")
print("=" * 52)
r, log = gcd(48, 18)
for l in log: print(f"  {l}")
print(f"  Result = {r}  (verify: math.gcd(48,18) = {math.gcd(48,18)})")
print()

print("=" * 52)
print("  FAST POWER: 2^10")
print("=" * 52)
r, log = fast_power(2, 10)
for l in log: print(f"  {l}")
print(f"  Result = {r}  (verify: 2^10 = {2**10})")
print()

print("=" * 52)
print("  BINARY SEARCH: find 36 in [0, 2, 4, ..., 48]")
print("=" * 52)
arr = list(range(0, 50, 2))
print(f"  Array : {arr}")
print()
r, log = binary_search(arr, 36)
for l in log: print(f"  {l}")
print(f"  Result = index {r}  (arr[{r}] = {arr[r] if r>=0 else 'N/A'})")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 3 — CALL STACK PROFILER
#  Wraps each function to count total calls and max recursion depth.
#  Shows dramatically how O(n) vs O(2^n) diverge as n grows.
# ═══════════════════════════════════════════════════════════════════════

class CallProfiler:
    """
    Transparent wrapper that counts calls and tracks max stack depth.
    Wraps any recursive function without modifying it.
    """
    def __init__(self, fn):
        self.fn        = fn
        self.calls     = 0
        self.max_depth = 0
        self._depth    = 0

    def __call__(self, *args, **kwargs):
        self.calls    += 1
        self._depth   += 1
        self.max_depth = max(self.max_depth, self._depth)
        result         = self.fn(*args, **kwargs)
        self._depth   -= 1
        return result

    def reset(self):
        self.calls = self.max_depth = self._depth = 0


# Wrap plain (non-logging) versions for profiling
def _fact(n):        return 1 if n <= 1 else n * _fact(n - 1)
def _fib(n):         return n if n <= 1 else _fib(n-1) + _fib(n-2)
def _fib_m(n, c=None):
    if c is None: c = {}
    if n in c: return c[n]
    if n <= 1: return n
    c[n] = _fib_m(n-1,c) + _fib_m(n-2,c); return c[n]

pf = CallProfiler(_fact); _fact  = pf
pb = CallProfiler(_fib);  _fib   = pb
pm = CallProfiler(_fib_m); _fib_m = pm

NS = list(range(1, 16))

print("=" * 70)
print("  CALL STACK PROFILER — Total Calls & Max Depth vs n")
print("=" * 70)
print(f"  {'n':>3}  {'factorial':>12}  {'depth':>7}"
      f"  {'fib_naive':>14}  {'depth':>7}  {'fib_memo':>10}  {'depth':>7}")
print("─" * 70)

for n in NS:
    pf.reset(); _fact(n)
    pb.reset(); _fib(n)
    pm.reset(); _fib_m(n)
    print(f"  {n:>3}  {pf.calls:>12,}  {pf.max_depth:>7}"
          f"  {pb.calls:>14,}  {pb.max_depth:>7}"
          f"  {pm.calls:>10,}  {pm.max_depth:>7}")

print("─" * 70)
print("  factorial  →  O(n)   calls — linear growth")
print("  fib naive  →  O(2^n) calls — doubles every step!")
print("  fib memo   →  O(n)   calls — cache eliminates redundancy")
print("=" * 70)


In [ ]:
# =========================================================
#  CELL 4 - GROWTH CHARTS
#  Chart A+B : Call count - linear and log scale
#  Chart C   : Tower of Hanoi move count = 2^n - 1
# =========================================================

ns_range = list(range(1, 17))

def _f2(n):        return 1 if n<=1 else n*_f2(n-1)
def _b2(n):        return n if n<=1 else _b2(n-1)+_b2(n-2)
def _m2(n, c=None):
    if c is None: c={}
    if n in c: return c[n]
    if n<=1: return n
    c[n]=_m2(n-1,c)+_m2(n-2,c); return c[n]

pF2 = CallProfiler(_f2); _f2 = pF2
pB2 = CallProfiler(_b2); _b2 = pB2
pM2 = CallProfiler(_m2); _m2 = pM2

fc2 = []; bc2 = []; mc2 = []
for n in ns_range:
    pF2.reset(); _f2(n); fc2.append(pF2.calls)
    pB2.reset(); _b2(n); bc2.append(pB2.calls)
    pM2.reset(); _m2(n); mc2.append(pM2.calls)

# -- Chart A+B: Linear and log scale ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Recursive Call Count Growth",
             fontsize=13, fontweight="bold", y=1.01)

for ax, logy in zip(axes, [False, True]):
    ax.plot(ns_range, fc2, "o-", color="#457B9D", lw=2.5, ms=7,
            label="Factorial  O(n)")
    ax.plot(ns_range, mc2, "s--", color="#2A9D8F", lw=2.5, ms=7,
            label="Fib Memoized  O(n)")
    ax.plot(ns_range, bc2, "^-", color="#E63946", lw=2.5, ms=7,
            label="Fib Naive  O(2^n)")
    if logy:
        ax.set_yscale("log")
        ax.set_title("Log Scale - exponential gap visible", pad=8)
    else:
        ax.set_title("Linear Scale", pad=8)
    ax.set_xlabel("n")
    ax.set_ylabel("Total Recursive Calls" + (" (log)" if logy else ""))
    ax.legend(fontsize=10, framealpha=0.9)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/p3_call_growth.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart A+B: call count growth - done")


# -- Chart C: Tower of Hanoi move count --------------------------------
def count_hanoi(n):
    moves = [0]
    def h(n, s, a, d):
        if n == 1: moves[0] += 1; return
        h(n-1, s, d, a); moves[0] += 1; h(n-1, a, s, d)
    h(n, "A", "B", "C")
    return moves[0]

disks  = list(range(1, 17))
actual = [count_hanoi(d) for d in disks]
theory = [2**d - 1 for d in disks]

fig2, ax2 = plt.subplots(figsize=(11, 5))
ax2.bar(disks, actual, color="#9B5DE5", alpha=0.85, label="Actual moves")
ax2.plot(disks, theory, "r--o", lw=1.8, ms=5, label="2^n - 1 formula")
ax2.set_xlabel("Number of Disks")
ax2.set_ylabel("Total Moves Required")
ax2.set_title("Tower of Hanoi - Move Count = 2^n - 1  (Exponential Growth)",
              pad=10)
ax2.legend(fontsize=10)
ax2.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/p3_hanoi_growth.png", bbox_inches="tight", dpi=110)
plt.show()
verified = actual == theory
print(f"Chart C: Hanoi growth - formula verified: {verified}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 5 — AUTOMATED TEST SUITE
#  Verifies every recursive function against known correct values.
# ═══════════════════════════════════════════════════════════════════════

REC_TESTS = [
    # description                 call                               expected
    ("factorial(0)",              lambda: factorial(0)[0],           1),
    ("factorial(1)",              lambda: factorial(1)[0],           1),
    ("factorial(5)",              lambda: factorial(5)[0],         120),
    ("factorial(10)",             lambda: factorial(10)[0],     3628800),
    ("fibonacci(0)",              lambda: fibonacci(0)[0],           0),
    ("fibonacci(1)",              lambda: fibonacci(1)[0],           1),
    ("fibonacci(8)",              lambda: fibonacci(8)[0],          21),
    ("fibonacci(10) memoized",    lambda: fibonacci(10, use_memo=True)[0], 55),
    ("fibonacci(12) memoized",    lambda: fibonacci(12, use_memo=True)[0],144),
    ("hanoi(1) = 1 move",         lambda: len(tower_of_hanoi(1)[0]),  1),
    ("hanoi(3) = 7 moves",        lambda: len(tower_of_hanoi(3)[0]),  7),
    ("hanoi(5) = 31 moves",       lambda: len(tower_of_hanoi(5)[0]), 31),
    ("hanoi(8) = 255 moves",      lambda: len(tower_of_hanoi(8)[0]),255),
    ("gcd(48, 18) = 6",           lambda: gcd(48, 18)[0],            6),
    ("gcd(100, 75) = 25",         lambda: gcd(100, 75)[0],          25),
    ("gcd(7, 0) = 7",             lambda: gcd(7, 0)[0],              7),
    ("fast_power(2, 10) = 1024",  lambda: fast_power(2, 10)[0],   1024),
    ("fast_power(3, 5) = 243",    lambda: fast_power(3, 5)[0],     243),
    ("fast_power(x, 0) = 1",      lambda: fast_power(7, 0)[0],       1),
    ("bsearch: found idx 18",     lambda: binary_search(list(range(0,50,2)), 36)[0], 18),
    ("bsearch: not found = -1",   lambda: binary_search(list(range(0,50,2)), 7)[0],  -1),
]

print("=" * 65)
print("  RECURSIVE FUNCTIONS — AUTOMATED TEST SUITE")
print("=" * 65)
total = passed = 0
for desc, fn, expected in REC_TESTS:
    total += 1
    try:
        result = fn()
        ok = (result == expected)
        if ok: passed += 1
        status = "✅" if ok else "❌"
        note = "" if ok else f"  ← got {result}"
        print(f"  {status}  {desc:<38} expected={expected:>8}{note}")
    except Exception as e:
        print(f"  ❌  {desc:<38} ERROR: {e}")

print("─" * 65)
print(f"  Total: {passed}/{total} tests passed")
if passed == total:
    print("  🎉 All recursive functions verified correct!")
print("=" * 65)
